In [3]:
#!/usr/bin/env python3
"""Archive 72 off-chain metadata docs -> metadata_raw_snapshot.jsonl (Colab, IPFS gateway fallback)."""
import os, json, base64, hashlib, datetime, urllib.parse
import pandas as pd, requests

DATA_DIR = "/content"                                   # upload agent_metadata.csv here
OUT      = "/content/metadata_raw_snapshot.jsonl"
TIMEOUT  = 30
IPFS_GATEWAYS = [
    "https://ipfs.io/ipfs/",
    "https://dweb.link/ipfs/",
    "https://cloudflare-ipfs.com/ipfs/",
    "https://gateway.pinata.cloud/ipfs/",
]
HEADERS = {"User-Agent": "erc8004-archive/1.0"}

def fetch_ipfs(cid):
    last = None
    for gw in IPFS_GATEWAYS:
        try:
            r = requests.get(gw + cid, timeout=TIMEOUT, headers=HEADERS)
            r.raise_for_status()
            return r.content, r.status_code, gw
        except Exception as e:
            last = e
    raise last

md = pd.read_csv(f"{DATA_DIR}/agent_metadata.csv")
now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

records, ok, fail = [], 0, 0
for _, row in md.iterrows():
    aid = int(row["agent_id"]); url = str(row["metadata_url_resolved"])
    rec = {"agent_id": aid, "resolved_url": url, "fetched_at": now}
    try:
        if url.startswith("data:"):
            header, _, data = url.partition(",")
            raw = base64.b64decode(data) if ";base64" in header else urllib.parse.unquote(data).encode("utf-8")
            rec["source_type"] = "data_uri_onchain"; rec["http_status"] = None
        elif "/ipfs/" in url:
            cid = url.split("/ipfs/")[1].split("/")[0].split("?")[0]
            raw, status, gw = fetch_ipfs(cid)
            rec["source_type"] = "ipfs"; rec["http_status"] = status
            rec["ipfs_cid"] = cid; rec["archived_via_gateway"] = gw
        else:
            r = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
            r.raise_for_status()
            raw = r.content; rec["source_type"] = "http"; rec["http_status"] = r.status_code
        rec["sha256"] = hashlib.sha256(raw).hexdigest()
        rec["content_bytes"] = len(raw)
        try:
            rec["raw_text"] = raw.decode("utf-8")
        except UnicodeDecodeError:
            rec["raw_base64"] = base64.b64encode(raw).decode("ascii")
        rec["status"] = "ok"; ok += 1
    except Exception as e:
        rec["status"] = "failed"; rec["error"] = repr(e)[:200]; fail += 1
    records.append(rec)

with open(OUT, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

by_type = {}
for rec in records:
    if rec["status"] == "ok":
        by_type[rec["source_type"]] = by_type.get(rec["source_type"], 0) + 1
print(f"wrote {OUT}")
print(f"archived OK: {ok} / {len(md)}  (by source: {by_type})  | unreachable: {fail}")
if fail:
    print("unreachable agents:", [(r['agent_id'], r['source_type'] if 'source_type' in r else '?') for r in records if r['status'] == 'failed'])

wrote /content/metadata_raw_snapshot.jsonl
archived OK: 70 / 72  (by source: {'http': 57, 'ipfs': 10, 'data_uri_onchain': 3})  | unreachable: 2
unreachable agents: [(6815, '?'), (6887, '?')]
